# Agente de IA para Análise de Vendas (Star Schema)

**Objetivo**: Agente conversacional que responde perguntas sobre vendas usando Nvidia + SQL Server (modelo dimensional).

**Arquitetura**:
- Interface: Jupyter (desenvolvimento) → Streamlit (produção)
- LLM: NVidia - nemotron-3-ultra-550b-a55b
- Banco: SQL Server (Star Schema)
- Processamento: Pandas + Plotly

**Autor**:  Roberto souza 
**Data**: 22/09/2026

## Imports e configuração

In [1]:
import os
import re
import json
from typing import Optional, Dict, Any, List
from datetime import datetime

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from openai import OpenAI
from sqlalchemy import create_engine, text

# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
SQL_CONNECTION_STRING = os.getenv("SQL_CONNECTION_STRING")

# Cliente NVIDIA
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

print("Chave NVIDIA carregada:", "Sim" if NVIDIA_API_KEY else "Não")
print("Connection String carregada:", "Sim" if SQL_CONNECTION_STRING else "Não")

Chave NVIDIA carregada: Sim
Connection String carregada: Sim


## Constantes e Schema permitido (segurança)

In [2]:
# Schema permitido – nunca deixe o LLM inventar tabelas/colunas
ALLOWED_TABLES = {
    "D_CLIENTE": ["COD_CLIENTE", "NOME", "NOME_FANTASIA", "CLASSIFICACAO_CLIENTE", "CIDADE", "ESTADO", "UF"],
    "D_EMPRESA": ["COD_EMPRESA", "NOME", "NOME_FANTASIA"],
    "D_PRODUTO": ["COD_PRODUTO", "DESCRICAO", "DESCRICAO_REDUZIDA", "FAMILIA", "SECAO", "GRUPO", "SUB_GRUPO", "MARCA"],
    "D_VENDEDOR": ["COD_VENDEDOR", "NOME"],
    "F_VENDAS": ["N_DOC", "COD_EMPRESA", "COD_PRODUTO", "COD_VENDEDOR", "COD_CLIENTE", 
                 "MOVIMENTO", "QUANTIDADE", "VENDA_BRUTA", "DESCONTO_TOTAL", "VENDA_LIQUIDA"]
}

# Relações (para o prompt do Gemini)
SCHEMA_DESCRIPTION = """
Star Schema de Vendas:

DIMENSÕES:
- D_CLIENTE (COD_CLIENTE PK): NOME, NOME_FANTASIA, CLASSIFICACAO_CLIENTE, CIDADE, ESTADO, UF
- D_EMPRESA (COD_EMPRESA PK): NOME, NOME_FANTASIA
- D_PRODUTO (COD_PRODUTO PK): DESCRICAO, DESCRICAO_REDUZIDA, FAMILIA, SECAO, GRUPO, SUB_GRUPO, MARCA
- D_VENDEDOR (COD_VENDEDOR PK): NOME

FATO:
- F_VENDAS: N_DOC, COD_EMPRESA, COD_PRODUTO, COD_VENDEDOR, COD_CLIENTE, MOVIMENTO (date),
            QUANTIDADE (int), VENDA_BRUTA (money), DESCONTO_TOTAL (money), VENDA_LIQUIDA (money)

Joins típicos:
F_VENDAS.COD_CLIENTE = D_CLIENTE.COD_CLIENTE
F_VENDAS.COD_EMPRESA = D_EMPRESA.COD_EMPRESA
F_VENDAS.COD_PRODUTO = D_PRODUTO.COD_PRODUTO
F_VENDAS.COD_VENDEDOR = D_VENDEDOR.COD_VENDEDOR
"""

## Conexão com o SQL Server

In [3]:
import urllib.parse
from sqlalchemy import create_engine

def get_engine():
    connection_string = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=localhost\\SQLEXPRESS;"
        "DATABASE=DW;"
        "Trusted_Connection=yes;"
    )

    params = urllib.parse.quote_plus(connection_string)

    return create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

In [4]:
try:
    engine = get_engine()

    with engine.connect() as conn:
        print("✅ SQLAlchemy + PyODBC conectado ao SQL Server!")

except Exception as e:
    print(f"❌ Erro: {e}")

✅ SQLAlchemy + PyODBC conectado ao SQL Server!


## Funções de segurança SQL

In [79]:
def is_safe_sql(sql: str) -> bool:
    """Valida se a query é segura (apenas SELECT/CTE e tabelas/colunas permitidas)."""
    sql_upper = sql.upper().strip()

    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE",
                 "EXEC", "EXECUTE", "CREATE", "GRANT", "REVOKE", "--", ";--"]
    if any(cmd in sql_upper for cmd in forbidden):
        return False

    if not (sql_upper.startswith("SELECT") or sql_upper.startswith("WITH")):
        return False

    # Nomes de CTE (WITH nome AS (...), outro_nome AS (...)) contam como
    # "tabelas" válidas só dentro desta query, além da whitelist normal
    cte_names = set(re.findall(r'(?:WITH|,)\s*(\w+)\s+AS\s*\(', sql_upper))

    for table in re.findall(r'\bFROM\s+(\w+)|\bJOIN\s+(\w+)', sql_upper):
        t = table[0] or table[1]
        if t and t not in ALLOWED_TABLES and t not in cte_names:
            return False

    return True

def execute_safe_query(sql: str) -> pd.DataFrame:
    """Executa query apenas se for segura."""
    if not is_safe_sql(sql):
        raise ValueError("Query rejeitada por segurança. Apenas SELECT com tabelas permitidas.")
    
    engine = get_engine()
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    return df

## Prompt engineering + Nvidia

In [26]:
## Prompt engineering + NVIDIA

def build_prompt(user_question: str) -> str:
    return f"""
Você é um analista de dados especializado em vendas.

Seu único trabalho é converter a pergunta do usuário em uma query SQL Server válida e segura.

REGRAS OBRIGATÓRIAS:

1. Responda APENAS com um JSON válido no formato:

{{
  "intention": "descrição curta da intenção",
  "sql": "SELECT ...",
  "needs_chart": true,
  "chart_type": "bar|line|pie|table|none",
  "explanation": "explicação curta do que a query faz"
}}

2. Use APENAS as tabelas e colunas existentes no schema fornecido.

3. Nunca use:
   INSERT
   UPDATE
   DELETE
   DROP
   ALTER
   TRUNCATE
   CREATE
   EXEC

4. A query deve ser somente SELECT.

5. Prefira agregações como:
   SUM
   COUNT
   AVG
   MIN
   MAX
   quando fizer sentido.

6. Use aliases claros e em português quando apropriado.

7. Para análises temporais, utilize a coluna MOVIMENTO.

8. Não invente tabelas, colunas ou relacionamentos que não estejam no schema.

9. Se a pergunta não puder ser respondida utilizando o schema fornecido:
   - sql deve ser null
   - needs_chart deve ser false
   - chart_type deve ser "none"
   - explique o motivo.

10. Escolha o gráfico de acordo com o tipo de análise:
   - line: evolução temporal
   - bar: comparação entre categorias
   - pie: participação percentual com poucas categorias
   - table: quando gráfico não fizer sentido
   - none: quando não houver necessidade de visualização

11. Não coloque markdown, comentários ou texto fora do JSON.

SCHEMA:
{SCHEMA_DESCRIPTION}

PERGUNTA DO USUÁRIO:
{user_question}
"""


def ask_nvidia(question: str) -> Dict[str, Any]:
    prompt = build_prompt(question)

    try:
        response = client.chat.completions.create(
            model="nvidia/nemotron-3-ultra-550b-a55b",
            messages=[
                {
                    "role": "system",
                    "content": "Você é um especialista em SQL Server e análise de dados."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=2000,
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            }
        )

        response_text = response.choices[0].message.content.strip()

        # Remove eventual bloco Markdown ```json ... ```
        response_text = re.sub(
            r"^```json\s*|\s*```$",
            "",
            response_text,
            flags=re.IGNORECASE
        ).strip()

        return json.loads(response_text)

    except json.JSONDecodeError:
        return {
            "intention": "erro de parsing",
            "sql": None,
            "needs_chart": False,
            "chart_type": "none",
            "explanation": response_text
        }

    except Exception as e:
        return {
            "intention": "erro na chamada da NVIDIA",
            "sql": None,
            "needs_chart": False,
            "chart_type": "none",
            "explanation": str(e)
        }

## Orquestrador principal

In [9]:
def create_chart(df: pd.DataFrame, chart_type: str, title: str, category_threshold: int = 8):
    """Cria gráfico Plotly básico e inteligente.
    
    category_threshold: acima desse número de categorias, bar vira horizontal
    para melhor leitura dos rótulos.
    """
    if df.empty:
        return None

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

    if not numeric_cols:
        return None

    if categorical_cols:
        x_col = categorical_cols[0]
        y_candidates = numeric_cols
    else:
        x_col = numeric_cols[0]
        y_candidates = numeric_cols[1:] or numeric_cols

    y_col = next((c for c in y_candidates if c != x_col), numeric_cols[0])

    n_categories = df[x_col].nunique()

    if chart_type == "pie":
        fig = px.pie(df.sort_values(by=x_col), names=x_col, values=y_col, title=title)

    elif chart_type == "line":
        fig = px.line(df.sort_values(by=x_col), x=x_col, y=y_col, title=title)

    else:  # bar (default)
        if n_categories > category_threshold:
            # horizontal, ordenado do maior pro menor (fica de cima pra baixo na leitura)
            df_sorted = df.sort_values(by=y_col, ascending=True)
            fig = px.bar(
                df_sorted, x=y_col, y=x_col, orientation="h", title=title
            )
            # altura dinâmica: mais categorias = gráfico mais alto, senão fica espremido
            fig.update_layout(height=max(400, n_categories * 28))
        else:
            df_sorted = df.sort_values(by=x_col)
            fig = px.bar(df_sorted, x=x_col, y=y_col, title=title)

    fig.update_layout(template="plotly_white")
    return fig

## Gerador de insights

In [44]:
def generate_insights(
    question: str,
    df: pd.DataFrame,
    intention: str = ""
) -> str:

    if df.empty:
        return "A consulta não retornou dados para análise."

    # Limita os dados enviados ao modelo
    df_analysis = df.head(100)

    data = df_analysis.to_json(
        orient="records",
        force_ascii=False
    )

    prompt = f"""
Você é um analista de dados sênior especializado em
análise de desempenho comercial e geração de insights
executivos..

Analise os dados abaixo e gere insights objetivos para ajudar
na interpretação dos resultados.

PERGUNTA DO USUÁRIO:
{question}

INTENÇÃO DA ANÁLISE:
{intention}

DADOS DA CONSULTA:
{data}

REGRAS:
- Baseie-se somente nos dados apresentados.
- Não invente informações.
- Identifique tendências, maiores valores, menores valores,
  diferenças relevantes e concentrações.
- Se houver evolução temporal, identifique crescimento ou queda.
- Quando possível, mencione os valores encontrados.
- Gere no máximo 5 insights.
- Seja objetivo, analítico e profissional.
- Escreva em português.
- Não utilize Markdown.
- Não utilize asteriscos (*).
- Não utilize emojis.
- Evite termos exagerados ou sensacionalistas como
  "abismal", "extremo", "absurdo", "alarmante" ou similares.
- Prefira linguagem executiva, clara e neutra.
- Não repita simplesmente os dados da tabela.
- Cada insight deve destacar uma conclusão relevante
  e explicar brevemente o que ela representa para a análise.

FORMATO:
1. Título curto: explicação objetiva.
2. Título curto: explicação objetiva.
3. Título curto: explicação objetiva.
"""

    try:

        response = client.chat.completions.create(
            model="nvidia/nemotron-3-ultra-550b-a55b",
            messages=[
                {
                    "role": "system",
                    "content": "Você é um analista de dados especializado em vendas."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=1000,
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            }
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"Não foi possível gerar os insights: {str(e)}"

## Interface interativa no Notebook

In [37]:
def chat(question: str):
    """Interface amigável para testar o agente."""

    result = run_agent(question)

    if not result["success"]:
        print(f"❌ {result['message']}")
        return

    print(f"\n✅ {result['explanation']}")
    print(f"📊 Linhas retornadas: {result['row_count']}")

    # Tabela
    display(result["data"].head(20))

    # Insights
    if result.get("insights"):
        print("\n💡 INSIGHTS")
        print("=" * 60)
        print(result["insights"])

    # Gráfico
    if result.get("fig"):
        print("\n📈 VISUALIZAÇÃO")
        result["fig"].show()

    # Não retorna o dicionário completo

## ORQUESTRAÇÃO DO AGENTE

In [81]:
import re
import time

def is_safe_sql(sql: str) -> tuple[bool, str]:
    """
    Valida se a query é segura: apenas SELECT/WITH, sem comandos perigosos,
    e apenas tabelas da whitelist (ou CTEs declaradas na própria query).
    Retorna (é_seguro, motivo_se_não_for).
    """
    sql_upper = sql.upper().strip()

    forbidden = [
        "INSERT", "UPDATE", "DELETE", "DROP", "ALTER",
        "TRUNCATE", "EXEC", "EXECUTE", "CREATE", "GRANT", "REVOKE",
        "--", ";--"
    ]
    for cmd in forbidden:
        if cmd in sql_upper:
            return False, f"A consulta contém um comando não permitido: {cmd}"

    if not (sql_upper.startswith("SELECT") or sql_upper.startswith("WITH")):
        return False, "A consulta gerada não é um SELECT (ou CTE) válido."

    # Nomes de CTE (WITH nome AS (...), outro_nome AS (...)) contam como
    # "tabelas" válidas só dentro desta query
    cte_names = set(re.findall(r'(?:WITH|,)\s*(\w+)\s+AS\s*\(', sql_upper))

    for match in re.findall(r'\bFROM\s+(\w+)|\bJOIN\s+(\w+)', sql_upper):
        table = match[0] or match[1]
        if table and table not in ALLOWED_TABLES and table not in cte_names:
            return False, f"Tabela não permitida na consulta: {table}"

    return True, ""


def run_agent(question: str, max_sql_attempts: int = 2) -> Dict[str, Any]:
    """
    Orquestra todo o processo:
    pergunta → NVIDIA → SQL → validação → SQL Server → DataFrame
    → insights + gráfico

    Se a execução falhar (erro de sintaxe/alias do SQL gerado), tenta
    corrigir automaticamente pedindo pro modelo revisar, até max_sql_attempts vezes.
    """
    try:
        # 1. NVIDIA interpreta a pergunta e gera o SQL
        analysis = ask_nvidia(question)
        sql = analysis.get("sql")

        if not sql:
            return {
                "success": False,
                "message": analysis.get(
                    "explanation",
                    "Não foi possível gerar uma consulta SQL."
                )
            }

        df = None
        last_error = None

        for attempt in range(max_sql_attempts):
            # 2. Validação de segurança (SELECT/WITH + whitelist de tabelas)
            is_safe, reason = is_safe_sql(sql)
            if not is_safe:
                return {
                    "success": False,
                    "message": reason,
                    "sql": sql,
                }

            # 3. Tenta executar no SQL Server
            try:
                with engine.connect() as conn:
                    df = pd.read_sql(text(sql), conn)
                break  # deu certo, sai do loop

            except Exception as e:
                last_error = str(e)

                if attempt == max_sql_attempts - 1:
                    return {
                        "success": False,
                        "message": f"Erro ao executar a consulta: {last_error}",
                        "sql": sql,
                    }

                # Pede pro modelo corrigir a própria query com o erro em mãos
                fix_prompt = f"""
A query SQL abaixo falhou ao ser executada no SQL Server, com o erro:
{last_error}

QUERY ORIGINAL:
{sql}

Corrija a query mantendo a mesma intenção da pergunta original: "{question}"
Responda APENAS com o JSON no mesmo formato de antes (intention, sql, needs_chart, chart_type, explanation).
"""
                analysis = ask_nvidia(fix_prompt)
                sql = analysis.get("sql")

                if not sql:
                    return {
                        "success": False,
                        "message": "Não foi possível corrigir a consulta automaticamente.",
                    }

        # 4. Gera insights
        insights = generate_insights(
            question=question,
            df=df,
            intention=analysis.get("intention", "")
        )

        # 5. Cria gráfico, se necessário
        fig = None
        if analysis.get("needs_chart", False):
            fig = create_chart(
                df=df,
                chart_type=analysis.get("chart_type", "bar"),
                title=analysis.get("intention", "Análise de vendas")
            )

        # 6. Retorna o resultado completo
        return {
            "success": True,
            "message": "Consulta executada com sucesso.",
            "question": question,
            "intention": analysis.get("intention"),
            "sql": sql,
            "explanation": analysis.get("explanation"),
            "data": df,
            "row_count": len(df),
            "insights": insights,
            "fig": fig
        }

    except Exception as e:
        return {
            "success": False,
            "message": f"Erro ao executar o agente: {str(e)}"
        }

## 🤖 INTERFACE DE INTERAÇÃO COM O AGENTE 🧠

In [87]:
chat("Qual o total de vendas líquidas por mês em 2019?")


✅ A query agrupa as vendas líquidas por mês do ano de 2019, ordenando cronologicamente para visualização da evolução temporal.
📊 Linhas retornadas: 7


,Mes,Total_Vendas_Liquidas
0,6,3569425.45
1,7,8182328.77
2,8,8899305.86
3,9,9211356.64
4,10,8278676.27
5,11,9046397.68
6,12,9091037.17



💡 INSIGHTS
1. Concentração no segundo semestre: Os dados disponíveis referem-se exclusivamente aos meses de junho a dezembro, indicando que o primeiro semestre de 2019 não está contemplado na base analisada.

2. Pico de faturamento em setembro: O mês de setembro registrou o maior volume de vendas líquidas do período, totalizando R$ 9,21 milhões, representando o auge da performance comercial no segundo semestre.

3. Crescimento sustentado até setembro: Houve evolução consistente das vendas entre junho (R$ 3,57 milhões) e setembro (R$ 9,21 milhões), com aumento acumulado de aproximadamente 158% no quadrimestre.

4. Estabilização em patamar elevado no trimestre final: Após o pico de setembro, as vendas oscilaram em faixa estreita entre R$ 8,28 milhões e R$ 9,09 milhões nos meses de outubro a dezembro, sugerindo consolidação da demanda em nível superior ao do meio do ano.

5. Junho como outlier de baixo desempenho: O valor de junho (R$ 3,57 milhões) destoa significativamente da média dos 

In [61]:
chat("Qual o total de vendas líquidas por mês em 2020?")


✅ A query soma a venda líquida agrupada por mês do ano de 2020, ordenada cronologicamente.
📊 Linhas retornadas: 10


,Mes,Total_Vendas_Liquidas
0,1,7604391.98
1,2,5202474.13
2,3,7399265.39
3,4,9136754.16
4,5,6774603.92
5,6,8299686.20
6,7,7618056.59
7,8,6395591.95
8,9,7524646.70
9,10,4180428.36



💡 INSIGHTS
1. Pico de vendas em abril: O mês de abril registrou o maior volume do período, com R$ 9,14 milhões, representando um crescimento de 23,5% em relação à média dos demais meses e indicando uma concentração sazonal relevante no segundo trimestre.

2. Queda acentuada em outubro: Outubro apresentou o menor resultado do ano (R$ 4,18 milhões), uma redução de 45,2% frente ao pico de abril e 37,8% abaixo da média mensal, sugerindo sazonalidade negativa ou evento atípico no último mês analisado.

3. Volatilidade trimestral significativa: O coeficiente de variação de 19,6% demonstra instabilidade no faturamento mensal, com oscilações superiores a R$ 2 milhões entre meses consecutivos, como a queda de março para abril e a recuperação de maio para junho.

4. Concentração no primeiro semestre: Os seis primeiros meses concentraram 62,3% do faturamento total do período (R$ 44,4 milhões de R$ 71,3 milhões), evidenciando desempenho mais forte na primeira metade do ano.

5. Recuperação parcia

In [88]:
chat("Quais os 10 produtos mais vendidos em quantidade em 2020?")


✅ A query agrega a quantidade vendida por produto no ano de 2020, ordena do maior para o menor e limita aos 10 primeiros. Gráfico de barras é ideal para comparar volumes entre categorias (produtos).
📊 Linhas retornadas: 10


,Produto,Quantidade_Total
0,Produto 660,353061
1,Produto 33577,299900
2,Produto 34593,169701
3,Produto 33696,89360
4,Produto 2824,81019
5,Produto 33832,76226
6,Produto 35529,73700
7,Produto 3159,70136
8,Produto 2414,59498
9,Produto 34709,57512



💡 INSIGHTS
1. Concentração no líder: O Produto 660 registrou 353.061 unidades, volume 18% superior ao segundo colocado (Produto 33577 com 299.900), indicando dependência significativa de um único item para puxar o volume total.

2. Queda acentuada após o topo: A diferença entre o primeiro e o terceiro produto (Produto 34593 com 169.701) é de 52%, demonstrando que a curva de demanda decai rapidamente fora dos dois primeiros colocados.

3. Grupo intermediário homogêneo: Os produtos da 4ª à 7ª posição (Produto 33696 a Produto 35529) apresentam volumes entre 89.360 e 73.700 unidades, com variação inferior a 18%, sugerindo um bloco de itens com desempenho de venda semelhante.

4. Disparidade entre extremidades do ranking: O décimo produto (Produto 34709 com 57.512) representa apenas 16% do volume do líder, evidenciando longa cauda de distribuição mesmo dentro do top 10.

5. Volume acumulado relevante: Os dez produtos somam 1.330.613 unidades vendidas, com os três primeiros respondendo por 

In [40]:
# Join com dimensão de cliente
chat("Quais os 5 estados com maior faturamento líquido em 2020?")


✅ A query junta a tabela de vendas com a dimensão de clientes para agrupar o faturamento líquido por estado (UF) no ano de 2020, ordenando do maior para o menor e limitando aos 5 primeiros.
📊 Linhas retornadas: 5


,Estado,Faturamento_Liquido
0,CE,32619991.38
1,RJ,6949075.62
2,MA,5193704.23
3,PE,4665260.43
4,AL,3651332.01



💡 INSIGHTS
• O Ceará (CE) lidera isoladamente o ranking com faturamento líquido de R$ 32,6 milhões, representando aproximadamente 68% do total dos 5 estados listados e superando em 4,7 vezes o segundo colocado (RJ).
• Existe uma concentração acentuada no Nordeste: 4 dos 5 estados (CE, MA, PE, AL) pertencem à região, respondendo juntos por cerca de 86% do faturamento total deste grupo.
• O Rio de Janeiro (RJ) é o único representante de fora do Nordeste no topo, ocupando a 2ª posição com R$ 6,9 milhões, porém com valor 53% inferior ao líder.
• A diferença entre o 1º e o 5º lugar é de quase 9 vezes (R$ 32,6 mi vs R$ 3,6 mi), evidenciando forte assimetria na distribuição de receita entre os estados de maior destaque.
• O faturamento acumulado dos 5 estados totaliza aproximadamente R$ 48,1 milhões, com o top 3 (CE, RJ, MA) concentrando 91% desse valor.

📈 VISUALIZAÇÃO


In [46]:
# Filtro por texto/categoria (testa geração de WHERE com string)
chat("Quais são 10 vendedores com maiores vendas ?")


✅ A query soma a VENDA_LIQUIDA por vendedor, ordena do maior para o menor e limita aos 10 primeiros. Gráfico de barras é ideal para comparar o desempenho entre eles.
📊 Linhas retornadas: 10


,Vendedor,Total_Venda_Liquida
0,Vendedor 16,45450285.90
1,Vendedor 4,21383278.75
2,Vendedor 3,14411746.64
3,Vendedor 1,14102553.58
4,Vendedor 15,4541177.49
5,Vendedor 5,4100392.96
6,Vendedor 17,3889677.70
7,Vendedor 20,3822690.12
8,Vendedor 18,3496210.86
9,Vendedor 2,3130316.18



💡 INSIGHTS
1. Concentração de receita no líder: O Vendedor 16 registrou R$ 45,45 milhões em vendas líquidas, valor 2,1 vezes superior ao segundo colocado (Vendedor 4, com R$ 21,38 milhões), indicando dependência significativa de um único executivo para o resultado agregado.

2. Lacuna expressiva no topo do ranking: A diferença de R$ 24,07 milhões entre o primeiro e o segundo colocado supera o faturamento somado dos vendedores da 3ª à 10ª posição (R$ 20,38 milhões), demonstrando assimetria acentuada de performance na camada de alta performance.

3. Formação de três tiers de desempenho: Os dados revelam estratificação clara: Tier 1 (Vendedor 16, > R$ 45 mi), Tier 2 (Vendedores 4, 3 e 1, entre R$ 14 mi e R$ 21 mi) e Tier 3 (demais sete vendedores, todos abaixo de R$ 4,6 mi), sugerindo necessidades distintas de gestão e incentivo por grupo.

4. Queda acentuada após a 4ª posição: O Vendedor 15 (5º lugar) apresenta R$ 4,54 milhões, representando redução de 68% em relação ao Vendedor 1 (4º l

In [47]:
chat("Quais os 5 produtos com maior faturamento líquido em 2019?")


✅ A query junta a tabela de vendas com a dimensão de produtos, filtra o ano de 2019 pela coluna MOVIMENTO, agrupa por descrição do produto somando a VENDA_LIQUIDA e ordena decrescentemente para trazer os top 5.
📊 Linhas retornadas: 5


,Produto,Faturamento_Liquido
0,Produto 34642,3620837.52
1,Produto 2414,3508138.54
2,Produto 3159,2939193.89
3,Produto 3066,2590288.25
4,Produto 34305,2519935.29



💡 INSIGHTS
1. Concentração no topo: O Produto 34642 lidera com faturamento líquido de R$ 3,62 milhões, superando o segundo colocado em aproximadamente R$ 112 mil, o que indica uma vantagem competitiva moderada na primeira posição.

2. Grupo de alto desempenho próximo: Os três primeiros produtos (34642, 2414 e 3159) formam um bloco distinto com faturamento acima de R$ 2,9 milhões cada, separando-se claramente dos dois últimos do ranking.

3. Queda acentuada após o terceiro lugar: Há uma redução de cerca de R$ 349 mil entre o terceiro (Produto 3159, R$ 2,94 milhões) e o quarto colocado (Produto 3066, R$ 2,59 milhões), marcando uma quebra de continuidade nos valores.

4. Faixa de variação controlada no top 5: A diferença entre o líder e o quinto colocado (Produto 34305, R$ 2,52 milhões) é de aproximadamente R$ 1,1 milhão, indicando que os cinco principais produtos operam em uma mesma ordem de grandeza de faturamento.

5. Ausência de dominância absoluta: Nenhum produto concentra uma parce

In [49]:
chat("Quais marcas teve  maior faturamento líquido em 2019?")


✅ A query junta a tabela de vendas com a dimensão de produtos, filtra o ano de 2019 pela coluna MOVIMENTO, agrupa por marca e soma a venda líquida, ordenando do maior para o menor faturamento.
📊 Linhas retornadas: 10


,MARCA,Faturamento_Liquido
0,Ge Healthcare Do Brasil,10659415.36
1,Eurofarma-Segmenta,7624744.26
2,Aspen-Cellofarma-Agila,6101368.69
3,Abl,4536048.62
4,Abbott Center,4486145.28
5,Grifols,4420374.54
6,Bayer,3982322.73
7,Mylan Brasil,2961672.64
8,Prati,1620810.39
9,Servier Do Brasil Ltda,1187201.95



💡 INSIGHTS
1. Liderança isolada da GE Healthcare: A marca registrou faturamento líquido de R$ 10,66 milhões, superando em 39,8% a segunda colocada (Eurofarma-Segmenta, R$ 7,62 milhões), o que indica forte concentração de receita no topo do ranking.

2. Formação de pelotão intermediário: Eurofarma-Segmenta, Aspen-Cellofarma-Agila e ABL compõem um grupo com faturamento entre R$ 4,5 milhões e R$ 7,6 milhões, representando a faixa de volume médio-alto do período.

3. Agrupamento competitivo na faixa dos R$ 4 milhões: Abbott Center, Grifols e Bayer apresentam valores muito próximos (R$ 3,98 milhões a R$ 4,49 milhões), sugerindo disputa acirrada por participação de mercado nesse estrato.

4. Queda acentuada após o 7º lugar: Mylan Brasil (R$ 2,96 milhões) inaugura um degrau de faturamento significativamente inferior aos líderes, com Prati (R$ 1,62 milhões) e Servier (R$ 1,19 milhões) fechando a lista em patamares progressivamente menores.

5. Concentração de receita nas três primeiras marcas

In [84]:
chat("Quais porcentagem de top 5 sub_grupo em relação ao faturamento líquido em 2019?")


✅ A query calcula o faturamento líquido por sub-grupo em 2019, obtém o total geral do ano e retorna os top 5 sub-grupos com suas respectivas participações percentuais no faturamento total.
📊 Linhas retornadas: 5


,SUB_GRUPO,Faturamento_Liquido,Percentual_Faturamento
0,Antibiótico,15035613.01,26.72
1,Contraste,10691915.36,19.00
2,Controlados,9563701.93,16.99
3,Hemoderivados,4443417.75,7.90
4,Hormonais,3763983.22,6.69



💡 INSIGHTS
1. Concentração no Antibiótico: O sub-grupo Antibiótico responde por 26,72% do faturamento líquido, representando mais de um quarto da receita total e superando em 7,7 pontos percentuais o segundo colocado.

2. Dominância do Top 3: Os três primeiros sub-grupos (Antibiótico, Contraste e Controlados) somam 62,71% do faturamento, indicando que a receita é fortemente ancorada em um portfólio restrito de categorias.

3. Queda acentuada após o Top 3: Há um gap de 9,09 pontos percentuais entre o terceiro (Controlados, 16,99%) e o quarto colocado (Hemoderivados, 7,90%), evidenciando uma quebra relevante na curva de contribuição.

4. Baixa representatividade da cauda do Top 5: Os dois últimos sub-grupos do ranking (Hemoderivados e Hormonais) contribuem juntos com apenas 14,59% do faturamento, menos da metade do volume gerado pelo líder isolado.

5. Disparidade entre líder e quinto colocado: O faturamento de Antibióticos é aproximadamente 4 vezes superior ao de Hormonais (5º lugar), 